In [ ]:
    ############    #############   Rate limiting   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.2 Backend Engineering
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Rate Limiting   #############   ##############   

 =>  Rate limiting protects a service (and your cost) from bursts -- one client sending
       too many requests, or a bug causing a retry storm, shouldn't be able to take down
       the service for everyone else.

 =>  Token bucket is the most common algorithm: a bucket holds up to N tokens, refills at a
       steady rate, and each request consumes one token -- no tokens left means the request
       is rejected (typically with HTTP 429).


<img src="images/token-bucket-rate-limit.png" alt="Token bucket rate limiting: requests consume tokens, refill over time, empty bucket returns 429">

In [ ]:
import time

class TokenBucket:
    def __init__(self, capacity: int, refill_per_sec: float):
        self.capacity = capacity
        self.tokens = capacity
        self.refill_per_sec = refill_per_sec
        self.last_refill = time.monotonic()

    def _refill(self):
        now = time.monotonic()
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_per_sec)
        self.last_refill = now

    def allow(self) -> bool:
        self._refill()
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

bucket = TokenBucket(capacity=3, refill_per_sec=0)  # no refill, to make the demo deterministic
for i in range(5):
    print(f"request {i+1}: {'allowed' if bucket.allow() else 'REJECTED (429)'}")


In [ ]:
 =>  The first 3 requests are allowed (the bucket started full); requests 4 and 5 are
       rejected because the bucket is empty and refill_per_sec=0 for this deterministic demo.

 =>  In production, set refill_per_sec > 0 (e.g. 5 tokens/sec) so legitimate traffic keeps
       flowing at a sustainable rate instead of being blocked until some fixed window resets.


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()
limiter = TokenBucket(capacity=2, refill_per_sec=0)

@app.get("/expensive")
def expensive_endpoint():
    if not limiter.allow():
        raise HTTPException(status_code=429, detail="rate limit exceeded, try again later")
    return {"result": "computed"}

client = TestClient(app)
for i in range(3):
    r = client.get("/expensive")
    print(i+1, r.status_code, r.json())


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Swap the hand-written TokenBucket for the 'slowapi' library (a real FastAPI rate
           limiting middleware) and reproduce the same 429 behavior.

 =>  [ ] Make the limiter per-client (keyed by API key or IP) instead of global -- one
           heavy client shouldn't exhaust the budget for everyone else.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  A single GLOBAL rate limit shared by every client -- one noisy client can starve all
       others; key limits per client/API-key/IP instead.

 =>  Storing bucket state in a single process's memory when running multiple app instances
       behind a load balancer -- each instance has its own bucket, so the real effective
       limit becomes (limit x instance count). Use a shared store (Redis) for multi-instance
       deployments.
